In [15]:
pip install traveltimepy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import geopandas as gpd
import os
import requests

This script serves the purpose of generating a nxp matrix where n = number of planning units and p = number of base schools. We seek to calculate the travel time driving distance from each planning unit's centroid to each base high school.

In [7]:
class distanceMatrix:
    def __init__(self, pu_path, schools_path):
        self.pu_path = pu_path
        self.schools_path = schools_path

    def load_data(self):
        self.schools = gpd.read_file(f'{os.getcwd()}/data/{self.schools_path}').to_crs('EPSG:4326')
        self.pu = gpd.read_file(f'{os.getcwd()}/data/{self.pu_path}').head(10).to_crs('EPSG:4326')

    def get_pu_centroids(self):
        self.pu['centroid'] = self.pu.geometry.centroid
        self.centroids = self.pu['centroid']

    def get_school_loc(self):
        self.school_loc = self.schools.geometry.apply(lambda geo: (geo.x, geo.y))

    def get_pu_loc(self):
        self.pu_loc = self.centroids.apply(lambda geo: (geo.x, geo.y))

    def get_locations(self): #necessary for api
        #construct dictionary of named points for api
        self.locations = []
        for idx, row in self.pu.iterrows():
            loc = row['centroid']
            self.locations.append({'id':'pu_'+str(row['pu_2324_84']), 'coords':{'lat':loc.x, 'lng':loc.y}})  #process location id as string, not int
        for idx, row in self.schools.iterrows():
            loc = row.geometry
            self.locations.append({'id':row['name'], 'coords':{'lat':loc.x, 'lng':loc.y}})

    def construct_matrix(self, app_id, api_key, travel_mode = 'driving', time = '2025-09-25T08:00:00Z'):
        self.departure_searches = []

        #construct departure searches
        for idx, row in self.pu.iterrows():
            self.departure_searches.append({
                'id': 'pu_'+str(row['pu_2324_84']),
                'departure_location_id': 'pu_'+str(row['pu_2324_84']),
                'arrival_location_ids': self.schools['name'].to_list(),
                'transportation': {'type': travel_mode},
                'departure_time': time,
                'travel_time': 3600,
                'properties': ['travel_time']
            })
        
        
        url = "https://api.traveltimeapp.com/v4/time-filter"
        headers = {
            'X-Application-Id': app_id,
            'X-Api-Key': api_key,
            'Content-Type': 'application/json'
        }

        payload = {
            "locations": self.locations,
            "departure_searches": self.departure_searches
        }

        response = requests.post(url, headers=headers, json=payload)
        response = requests.post(url, headers=headers, json=payload)

        print("Status code:", response.status_code)

        try:
            print("Response JSON:", response.json())
        except Exception:
            print("Response content:", response.text)



In [8]:
matrix = distanceMatrix('pu_2324_SPLIT.geojson', 'dps_hs_locations.geojson')
matrix.load_data()
matrix.get_pu_centroids()
matrix.get_school_loc()
matrix.get_pu_loc()
matrix.get_locations()

C:\Users\olubl\AppData\Local\Temp\ipykernel_1604\4215406063.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  self.pu['centroid'] = self.pu.geometry.centroid


In [9]:
matrix.construct_matrix(app_id='a19380dd', api_key='f6af8ab6f3b957f478a041a5d7bf45f8')

Status code: 429
Response JSON: {'http_status': 429, 'error_code': 9, 'description': 'You have exceeded the maximum number of requests per minute allowed on your plan. To upgrade your plan to one with higher limits, log in to your account at https://account.traveltime.com and click Upgrade.', 'documentation_link': 'https://docs.traveltime.com/reference/error-codes', 'additional_info': {}}
